In [ ]:
import pandas as pd
import numpy as np

from scipy.stats import spearmanr, zscore
from statsmodels.stats.multitest import multipletests

In [ ]:
expr = pd.read_csv(
    "data/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    index_col=0
)

models = pd.read_csv("data/Model.csv")

prism = pd.read_csv(
    "data/secondary-screen-dose-response-curve-parameters.csv"
)

treatments = pd.read_csv(
    "data/secondary-screen-replicate-collapsed-treatment-info.csv"
)

cell_info = pd.read_csv(
    "data/secondary-screen-cell-line-info.csv"
)

In [ ]:



treatment_hypotheses = {
    "CDK4/6": {
        "targets": ["CDK4", "CDK6"],
        "drugs": ["palbociclib", "ribociclib", "abemaciclib"],
    },
    "WEE1": {
        "targets": ["WEE1"],
        "drugs": ["MK-1775", "AZD1775", "adavosertib", "azenosertib", "ZN-c3", "Debio 0123"],
    },
    "CHK1": {
        "targets": ["CHEK1"],
        "drugs": ["LY2603618", "SCH-900776", "MK-8776", "prexasertib"],
    },
    "SLC7A11/ferroptosis": {
        "targets": ["SLC7A11"],
        "drugs": [
            "sulfasalazine", "erastin", "imidazole ketone erastin",
            "RSL3", "ML162", "FIN56", "FINO2"
        ],
    },
    "PI3K": {
        "targets": ["PIK3CA", "PIK3R1", "PIK3R3"],
        "drugs": ["alpelisib", "pictilisib", "buparlisib", "taselisib", "copanlisib"],
    },
    "AKT": {
        "targets": ["AKT1", "AKT2", "AKT3"],
        "drugs": ["capivasertib", "ipatasertib", "MK-2206"],
    },
    "mTOR": {
        "targets": ["MTOR", "RPTOR"],
        "drugs": [
            "everolimus", "temsirolimus", "sirolimus", "ridaforolimus",
            "AZD8055", "Torin 1", "Torin 2"
        ],
    },
}


drug_annotation = {
    drug.lower(): hypothesis
    for hypothesis, config in treatment_hypotheses.items()
    for drug in config["drugs"]
}

requested_drugs = [
    drug
    for config in treatment_hypotheses.values()
    for drug in config["drugs"]
]


def gene_symbol(col):
    return str(col).split(" ")[0].upper()


expr = expr.rename(columns=lambda col: gene_symbol(col))


In [ ]:

prism["name_lower"] = prism["name"].astype(str).str.lower()

available_names = set(prism["name_lower"].dropna())

drug_availability = pd.DataFrame({
    "hypothesis": [drug_annotation[d.lower()] for d in requested_drugs],
    "requested_drug": requested_drugs,
    "available_in_PRISM": [d.lower() in available_names for d in requested_drugs],
})

drug_availability.sort_values(["hypothesis", "requested_drug"])


In [ ]:
print("Expression matrix:", expr.shape)
print("PRISM rows:", len(prism))
print("Available requested drugs:", drug_availability["available_in_PRISM"].sum())


In [ ]:
def build_mss_tmbh_score(
    expr,
    model_ids,
    de,
    gene_col="gene",
    lfc_col="log2FoldChange",
    padj_col="padj",
    padj_cutoff=0.05,
    lfc_cutoff=1,
    top_n=50,
    exclude_genes=None,
):
    """Build an MSS/TMB-H transcriptional similarity score.

    Score = mean(z-scored upregulated genes) - mean(z-scored downregulated genes)
    Higher values indicate greater similarity to the TCGA MSS/TMB-H profile.
    """
    exclude_genes = {str(g).upper() for g in (exclude_genes or [])}

    sig = de[
        (de[padj_col] < padj_cutoff) &
        (de[lfc_col].abs() >= lfc_cutoff)
    ].copy()
    sig[gene_col] = sig[gene_col].astype(str).str.upper()
    sig = sig[~sig[gene_col].isin(exclude_genes)]

    up = sig[sig[lfc_col] > 0].sort_values(lfc_col, ascending=False).head(top_n)
    down = sig[sig[lfc_col] < 0].sort_values(lfc_col, ascending=True).head(top_n)

    up_genes = up[gene_col].tolist()
    down_genes = down[gene_col].tolist()

    up_available = [g for g in up_genes if g in expr.columns]
    down_available = [g for g in down_genes if g in expr.columns]

    print(f"UP genes available: {len(up_available)}/{len(up_genes)}")
    print(f"DOWN genes available: {len(down_available)}/{len(down_genes)}")

    if not up_available or not down_available:
        raise ValueError("No usable up- or downregulated genes were found in DepMap.")

    x = expr.loc[
        expr.index.intersection(set(model_ids)),
        up_available + down_available,
    ].apply(pd.to_numeric, errors="coerce")

    
    x_z = x.apply(lambda col: zscore(col, nan_policy="omit"), axis=0)

    score = x_z[up_available].mean(axis=1) - x_z[down_available].mean(axis=1)
    score_df = score.rename("MSS_TMBH_score").reset_index()
    score_df.columns = ["DepMap_ID", "MSS_TMBH_score"]

    return score_df, up_available, down_available


In [ ]:
crc_models = models[
    models["OncotreeCode"] ==
    "COAD"
].copy()

print("CRC models:", len(crc_models))

In [ ]:
ucec_models = models[
    models["OncotreeCode"] == "UCEC"
].copy()

print("Uterus models:", len(ucec_models))

In [ ]:
de_crc = pd.read_csv(
    "data/DE_results/MSS_TMB_H_vs_MSS_TMB_L.csv", 
    index_col=0
)
de_crc["gene"] = de_crc.index.astype(str).str.upper()
de_crc["log2FoldChange"] = de_crc["logFC"].astype(float)
de_crc["padj"] = de_crc["FDR"].astype(float)
de_crc

In [ ]:
de_ucec = pd.read_csv(
    "data/DE_results/ucec/MSS_TMB_H_vs_MSS_TMB_L.csv", 
    index_col=0
)
de_ucec["gene"] = de_ucec.index.astype(str).str.upper()
de_ucec["log2FoldChange"] = de_ucec["logFC"].astype(float)
de_ucec["padj"] = de_ucec["FDR"].astype(float)
de_ucec

In [ ]:
expr.index = expr["MODELID"]
expr

In [ ]:
signature_excluded_genes = sorted({
    gene
    for config in treatment_hypotheses.values()
    for gene in config["targets"]
} | {"SLC3A2", "GPX4", "GCLC", "GCLM", "NFE2L2", "KEAP1", "AIFM2", "ACSL4", "TFRC", "GOT1"})

crc_score, crc_up, crc_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=crc_models["ModelID"],
    de=de_crc,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

crc_score.head()


In [ ]:
ucec_score, ucec_up, ucec_down = build_mss_tmbh_score(
    expr=expr,
    model_ids=ucec_models["ModelID"],
    de=de_ucec,
    top_n=50,
    exclude_genes=signature_excluded_genes,
)

ucec_score.head()


In [ ]:
def test_signature_drug_response(
    score_df,
    prism,
    treatment_hypotheses,
    response_col="auc",
    min_n=5,
):
    """Correlate the MSS/TMB-H score with PRISM drug response.

    For AUC, rho < 0 means higher MSS/TMB-H similarity is associated with
    greater drug sensitivity.
    """
    results = []

    for hypothesis, config in treatment_hypotheses.items():
        for requested_drug in config["drugs"]:
            drug_df = prism[
                prism["name"].astype(str).str.lower() == requested_drug.lower()
            ].copy()

            if drug_df.empty:
                continue

            df = score_df.merge(
                drug_df[["depmap_id", "name", response_col]],
                left_on="DepMap_ID",
                right_on="depmap_id",
                how="inner",
            )
            df = df.replace([np.inf, -np.inf], np.nan)
            df = df.dropna(subset=["MSS_TMBH_score", response_col])

            if len(df) < min_n:
                continue

            rho, p = spearmanr(df["MSS_TMBH_score"], df[response_col])
            results.append({
                "hypothesis": hypothesis,
                "drug": drug_df["name"].iloc[0],
                "analysis": "MSS/TMB-H signature",
                "n": len(df),
                "rho": rho,
                "pvalue": p,
            })

    results = pd.DataFrame(results)
    if not results.empty:
        results["FDR_global"] = multipletests(results["pvalue"], method="fdr_bh")[1]
        results["FDR_within_hypothesis"] = (
            results.groupby("hypothesis")["pvalue"]
            .transform(lambda p: multipletests(p, method="fdr_bh")[1])
        )
        results = results.sort_values("pvalue")

    return results


def test_target_expression_drug_response(
    expr,
    model_ids,
    prism,
    treatment_hypotheses,
    response_col="auc",
    min_n=5,
):
    """Test whether expression of each proposed target predicts response.

    For AUC, rho < 0 means higher target expression is associated with
    greater drug sensitivity.
    """
    results = []
    model_ids = set(model_ids)

    for hypothesis, config in treatment_hypotheses.items():
        for target in config["targets"]:
            if target not in expr.columns:
                print(f"Expression unavailable: {target}")
                continue

            target_expr = (
                expr.loc[expr.index.intersection(model_ids), [target]]
                .reset_index()
                .rename(columns={expr.index.name or "index": "DepMap_ID"})
            )
            if "DepMap_ID" not in target_expr.columns:
                target_expr = target_expr.rename(columns={target_expr.columns[0]: "DepMap_ID"})

            for requested_drug in config["drugs"]:
                drug_df = prism[
                    prism["name"].astype(str).str.lower() == requested_drug.lower()
                ].copy()
                if drug_df.empty:
                    continue

                df = target_expr.merge(
                    drug_df[["depmap_id", "name", response_col]],
                    left_on="DepMap_ID",
                    right_on="depmap_id",
                    how="inner",
                )
                df = df.replace([np.inf, -np.inf], np.nan)
                df[target] = pd.to_numeric(df[target], errors="coerce")
                df = df.dropna(subset=[target, response_col])

                if len(df) < min_n:
                    continue

                rho, p = spearmanr(df[target], df[response_col])
                results.append({
                    "hypothesis": hypothesis,
                    "target": target,
                    "drug": drug_df["name"].iloc[0],
                    "analysis": "target expression",
                    "n": len(df),
                    "rho": rho,
                    "pvalue": p,
                })

    results = pd.DataFrame(results)
    if not results.empty:
        results["FDR_global"] = multipletests(results["pvalue"], method="fdr_bh")[1]
        results["FDR_within_hypothesis"] = (
            results.groupby("hypothesis")["pvalue"]
            .transform(lambda p: multipletests(p, method="fdr_bh")[1])
        )
        results = results.sort_values("pvalue")

    return results


In [ ]:
crc_signature_results = test_signature_drug_response(
    score_df=crc_score,
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

crc_target_results = test_target_expression_drug_response(
    expr=expr,
    model_ids=crc_models["ModelID"],
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

crc_signature_results


In [ ]:
ucec_signature_results = test_signature_drug_response(
    score_df=ucec_score,
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_target_results = test_target_expression_drug_response(
    expr=expr,
    model_ids=ucec_models["ModelID"],
    prism=prism,
    treatment_hypotheses=treatment_hypotheses,
)

ucec_signature_results


In [ ]:
def add_cancer(df, cancer):
    out = df.copy()
    out.insert(0, "cancer", cancer)
    return out

all_signature_results = pd.concat(
    [
        add_cancer(crc_signature_results, "CRC"),
        add_cancer(ucec_signature_results, "UCEC"),
    ],
    ignore_index=True,
)

all_target_results = pd.concat(
    [
        add_cancer(crc_target_results, "CRC"),
        add_cancer(ucec_target_results, "UCEC"),
    ],
    ignore_index=True,
)

print("Signature-level tests")
display(all_signature_results.sort_values(["cancer", "pvalue"]))

print("Direct target-expression tests")
display(all_target_results.sort_values(["cancer", "pvalue"]))

all_signature_results.to_csv("all_signature_drug_results.csv", index=False)
all_target_results.to_csv("all_target_expression_drug_results.csv", index=False)
drug_availability.to_csv("prism_drug_availability.csv", index=False)


In [ ]:
robustness_results = []

for cancer, cancer_models, de in [
    ("CRC", crc_models, de_crc),
    ("UCEC", ucec_models, de_ucec),
]:
    for top_n in [20, 50, 100]:
        score_df, _, _ = build_mss_tmbh_score(
            expr=expr,
            model_ids=cancer_models["ModelID"],
            de=de,
            top_n=top_n,
            exclude_genes=signature_excluded_genes,
        )
        tmp = test_signature_drug_response(
            score_df=score_df,
            prism=prism,
            treatment_hypotheses=treatment_hypotheses,
        )
        tmp.insert(0, "cancer", cancer)
        tmp["top_n"] = top_n
        robustness_results.append(tmp)

robustness_results = pd.concat(robustness_results, ignore_index=True)
robustness_results.sort_values(["hypothesis", "drug", "cancer", "top_n"])



In [ ]:
def test_crispr_dependency(score_df, gene_effect, gene="GOT1", min_n=5):
    """Correlate MSS/TMB-H score with a DepMap CRISPR gene-effect column."""
    if gene not in gene_effect.columns:
        raise KeyError(f"{gene} not found in gene-effect matrix")

    dep = gene_effect[[gene]].reset_index()
    dep.columns = ["DepMap_ID", "gene_effect"]
    df = score_df.merge(dep, on="DepMap_ID", how="inner").dropna()

    if len(df) < min_n:
        raise ValueError(f"Too few matched models: n={len(df)}")

    rho, p = spearmanr(df["MSS_TMBH_score"], df["gene_effect"])
    return pd.DataFrame([{
        "gene": gene,
        "n": len(df),
        "rho": rho,
        "pvalue": p,
        "interpretation": "rho < 0 means higher MSS/TMB-H score is associated with stronger dependency",
    }])








In [ ]:
models.loc[
    models["OncotreePrimaryDisease"]
    .astype(str)
    .str.contains("Colorectal|Rectal|Colon", case=False, na=False),
    ["OncotreeCode"]
].drop_duplicates()

In [ ]:
models.loc[
    models["OncotreePrimaryDisease"]
    .astype(str)
    .str.contains("Endometrial|Uterine", case=False, na=False),
    ["OncotreeCode"]
].drop_duplicates()